# The Anatomy of AI · Lecture 29 · Taking Apart the Weights — the notebook

Every tool in the course so far builds a **second network** to read the first: a probe, a prism, a transcoder. This
lecture takes apart the *weights themselves*. **Stochastic parameter decomposition** (SPD; Bushnaq, Braun and Sharkey,
2025) splits each weight matrix into simple rank-one **pieces** that add back up to it, and asks, input by input, which
pieces the output actually needs.

You will:

1. train Lecture 15's **pentagon** (5 features squeezed into 2 neurons) and follow one input through it;
2. write SPD from scratch, train it, and watch 20 random pieces become **5 survivors, one per feature**;
3. use the lecture's own trained pieces for surgery: switch one off and see exactly what it carried;
4. put a **prism** on the same two neurons and see why it finds something else;
5. read the lecture's bigger runs: 40 features in 10 neurons, an identity layer, and 100 jobs in 50 neurons.

Every key number is checked against the lecture's: a ✓ means you got what the lecture got. The tiny models run on the
CPU, exactly as the lecture ran them; SPD's 40,000 training steps run on the GPU (*Runtime → Change runtime type → T4
GPU*). About 10 minutes on free Colab.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, os, subprocess, sys
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
print("installed")

In [ ]:
import io, json, math, os, random, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

SITE = "https://glass-box-sigma.vercel.app/lectures/29-taking-apart-the-weights/"
DATA = {n: json.load(open(fetch(SITE + f"data/{n}.json", f"{n}.json", f"the lecture's {n} data")))
        for n in ("tms52_pieces", "tms52id", "tms4010", "resid", "sweep")}
torch.set_num_threads(2)

## Part the First: ten numbers

### Back to the pentagon (Plate II)

The model of Lecture 15: five features, each present 5% of the time with a random strength in [0, 1], squeezed into two
neurons and read back out:

$$\text{out} = \mathrm{ReLU}(W^\top W x + b), \qquad W \text{ is } 2 \times 5.$$

Trained exactly as the lecture trained it (same seed, same steps, on the CPU), it puts the five features' arrows (the
columns of $W$) at the corners of a pentagon.

In [ ]:
def batch(bs, n=5, p=0.05, device="cpu"):
    """Sparse inputs: each feature is on with probability p, at a strength drawn from [0, 1]."""
    v = torch.rand(bs, n, device=device)
    return v * (torch.rand(bs, n, device=device) < p)

torch.manual_seed(0)
W = torch.nn.Linear(5, 2, bias=False).weight.detach().clone().requires_grad_(True)   # 2 x 5: column j = feature j
b = torch.zeros(5, requires_grad=True)
tms = lambda x, W, b: F.relu(x @ W.T @ W + b)
opt = torch.optim.AdamW([W, b], lr=5e-3, weight_decay=0.01)
t = time.time()
for step in range(10000):
    x = batch(1024)
    loss = ((x - tms(x, W, b)) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
W, b = W.detach(), b.detach()
with torch.no_grad():
    xe = batch(200000)
    mse, zero = ((xe - tms(xe, W, b)) ** 2).mean().item(), (xe ** 2).mean().item()
ang = torch.rad2deg(torch.atan2(W[1], W[0]))
print(f"trained in {time.time() - t:.0f} s; error {mse:.5f} (predicting zero: {zero:.5f})")
print("arrow angles:", [f"{a:.1f}°" for a in ang.tolist()], " lengths:", [round(v, 3) for v in W.norm(dim=0).tolist()])
print("biases:", [round(v, 3) for v in b.tolist()])
W_lec, b_lec = torch.tensor([[0.807889997959137, 1.008239984512329, -0.4979099929332733, -0.19214999675750732, -1.1173700094223022], [0.7984099984169006, -0.5239400267601013, 1.0100699663162231, -1.1243300437927246, -0.16631999611854553]]), torch.tensor([-0.2376600056886673, -0.22721999883651733, -0.23853999376296997, -0.23982000350952148, -0.2345300018787384])
check("largest difference from the lecture's W", (W - W_lec).abs().max().item(), 0, tol=0.01)
check("the model's error", mse, 0.0017668812070041895, tol=1e-4)

fig, ax = plt.subplots(figsize=(4, 4))
for j in range(5):
    ax.annotate("", W[:, j].tolist(), (0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.4))
    ax.text(*(W[:, j] * 1.13).tolist(), f"{j}", ha="center", va="center")
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal"); ax.axhline(0, color=INK2, lw=0.4); ax.axvline(0, color=INK2, lw=0.4)
ax.set_title("the five features' arrows (columns of W)", fontsize=10, loc="left"); plt.show()
W, b = W_lec, b_lec          # from here on, the lecture's own numbers (the same model, to the third decimal)

### Follow one input (Plates III–IV)

Feature 0 at 0.8 and feature 2 at 0.6. The two neurons hold $h = Wx$; the output reads $W^\top h + b$ and clips at 0.
Feature 2's arrow is a neighbour of feature 0's, so some of it **leaks** into output 0.

In [ ]:
x = torch.tensor([0.8, 0, 0.6, 0, 0])
h = W @ x
out = F.relu(W.T @ h + b)
G = W.T @ W                                 # 5 x 5: how much each feature's arrow overlaps each other's
print("h =", [round(v, 3) for v in h.tolist()])
print("output =", [round(v, 4) for v in out.tolist()])
print(f"output 0 = 0.8 x {G[0, 0]:.3f} + 0.6 x {G[0, 2]:.3f} + ({b[0]:.3f}) = {0.8 * G[0, 0] + 0.6 * G[0, 2] + b[0]:.3f}")
print("the whole model is these ten numbers:\n", W.numpy().round(3))
check("the running input's output", [round(v, 3) for v in out.tolist()], [1.037, 0.0, 0.846, 0.0, 0.0])
check("feature 2's leak into output 0 (0.6 x its overlap)", 0.6 * G[0, 2].item(), 0.243, tol=1e-3)

### One piece, and pieces that add up (Plates V–VI)

A **piece** is a rank-one matrix: an outer product $u v^\top$. It *reads* the input along $v$ and *writes* along $u$. Any
matrix is a sum of pieces, in many ways. One obvious way for $W$: five pieces, one per column, each reading one feature
and writing its arrow. SPD's question is which split the *model's behaviour* asks for, without being told the features.

In [ ]:
pieces = [torch.outer(W[:, j], torch.eye(5)[j]) for j in range(5)]      # piece j: reads feature j, writes its arrow
print("the five column pieces add back up to W:", torch.allclose(sum(pieces), W))
U, S, Vt = torch.linalg.svd(W, full_matrices=False)
svd_pieces = [S[i] * torch.outer(U[:, i], Vt[i]) for i in range(2)]   # another split: two pieces (the SVD)
print("so do the two SVD pieces:", torch.allclose(sum(svd_pieces), W, atol=1e-6))
check("both splits rebuild W", bool(torch.allclose(sum(pieces), W) and torch.allclose(sum(svd_pieces), W, atol=1e-6)), True)

Three wishes for the split (Plate VII): **faithful** (the pieces add up to $W$), **minimal** (each input needs as few
pieces as possible), **simple** (each piece is rank one). The first method, APD (2025), ranked pieces by gradient
attribution; on this model its pieces came out about 11% short (its paper's numbers: MMCS 0.998, ML2R 0.893). SPD
replaced the attribution with a learned, tested **causal importance**.

---
## Part the Second: stochastic parameter decomposition

### How much of you do I need? (Plates IX–XI)

Give each weight matrix $C = 20$ pieces. For each input, each piece gets a **causal importance** between 0 and 1, from
a tiny network of its own (16 GELU units) that sees only how strongly the piece *reads* the input. Then the test (Plate
X): scale every piece by a random number between its importance and 1, run the model on the pieces, and demand the
original output. A piece with importance 1 must stay; a piece with importance 0 can be turned down to anything. Four
losses (Plate XI):

- **faithfulness**: the 20 pieces add up to $W$;
- **stochastic reconstruction**: the randomly turned-down pieces still give the right output, all layers at once;
- **layerwise reconstruction**: the same, one layer at a time;
- **importance**: the sum of importances, so that as few pieces as possible are needed (weight β = 3 × 10⁻³).

Here it is, from scratch, with the paper's settings. Layer 2 of the model is $W^\top$, so its pieces are layer 1's,
transposed: one set of 20 pieces serves both.

In [ ]:
GPU = "cuda" if torch.cuda.is_available() else "cpu"
Wg, bg = W.to(GPU), b.to(GPU)
C, DG, BETA, STEPS, BS = 20, 16, 3e-3, 40000, 4096
torch.manual_seed(1000)

# 20 random pieces A[:, c] (what piece c reads, 5 numbers) and B[c] (what it writes, 2 numbers), started at the size
# of W they already cover
A = torch.randn(5, C, device=GPU); A /= A.norm(dim=0, keepdim=True)
B = torch.randn(C, 2, device=GPU); B /= B.norm(dim=1, keepdim=True)
B *= torch.einsum("ic,co,oi->c", A, B, Wg)[:, None]
A.requires_grad_(True); B.requires_grad_(True)

def gate_net():   # one tiny network per piece: 1 number in, 16 GELUs, 1 number out
    return [(torch.randn(C, DG, device=GPU) * math.sqrt(2)).requires_grad_(True), torch.zeros(C, DG, device=GPU, requires_grad=True),
            (torch.randn(C, DG, device=GPU) / math.sqrt(DG)).requires_grad_(True), torch.zeros(C, device=GPU, requires_grad=True)]
gates = {"layer 1": gate_net(), "layer 2": gate_net()}

def importance(layer, reads):
    """Causal importance of each piece, from how strongly it reads its input. Two clippings of one number: 'lower' is
    used for the masks (leaky below 0, capped at 1), 'upper' in the penalty."""
    w_in, b_in, w_out, b_out = gates[layer]
    z = (F.gelu(reads[..., None] * w_in + b_in) * w_out).sum(-1) + b_out
    lower = torch.where(z > 0, z.clamp(max=1), 0.01 * z)
    upper = torch.where(z > 1, 1 + 0.01 * (z - 1), F.relu(z))
    return lower, upper

def run(x, masks=None, on=("layer 1", "layer 2")):
    """The model with the chosen layers built from pieces, each piece scaled by its mask. Also returns each layer's input."""
    ins, a = {}, x
    for layer in ("layer 1", "layer 2"):
        ins[layer] = a
        rd, wr = (A, B) if layer == "layer 1" else (B.T, A.T)            # layer 2 = layer 1 transposed
        if layer in on:
            c = a @ rd
            a = (c * masks[layer] if masks is not None else c) @ wr
        else:
            a = a @ (Wg.T if layer == "layer 1" else Wg)
    return F.relu(a + bg), ins

params = [A, B] + gates["layer 1"] + gates["layer 2"]
opt = torch.optim.AdamW(params, lr=1e-3, weight_decay=0.0)
history = []
t = time.time()
for step in range(STEPS + 1):
    lr = 1e-3 * math.cos(0.5 * math.pi * step / (STEPS - 1)) if step < STEPS else 0.0     # cosine decay to 0
    for g in opt.param_groups:
        g["lr"] = lr
    x = batch(BS, device=GPU)
    with torch.no_grad():
        target, _ = run(x, on=())
    _, ins = run(x, on=())
    lo, up = {}, {}
    for layer, rd in (("layer 1", A), ("layer 2", B.T)):
        lo[layer], up[layer] = importance(layer, ins[layer] @ rd)
    faith = (((A @ B).T - Wg) ** 2).sum() * 2 / 20                         # both layers, per weight
    masks = {k: lo[k] + (1 - lo[k]) * torch.rand_like(lo[k]) for k in lo}  # between importance and 1
    stoch = ((run(x, masks)[0] - target) ** 2).mean()
    layerwise = sum(((run(x, {k: lo[k] + (1 - lo[k]) * torch.rand_like(lo[k])}, on=(k,))[0] - target) ** 2).mean()
                    for k in lo) / 2
    imp = sum(up[k].sum(-1).mean() for k in up)
    loss = faith + stoch + layerwise + BETA * imp
    if step < STEPS:
        opt.zero_grad(); loss.backward(); opt.step()
    if step % 2000 == 0:
        alive = int((lo["layer 1"] > 0.1).any(0).sum())
        history.append((step, alive, faith.item(), stoch.item(), imp.item()))
print(f"SPD trained in {time.time() - t:.0f} s")
for s, a, f, r, i in history[::4] + (history[-1:] if (len(history) - 1) % 4 else []):
    print(f"step {s:6d}: pieces needed by some input {a:2d}; faithfulness {f:.1e}; reconstruction {r:.1e}; importance {i:.3f}")

### Watch it learn, and the five survivors (Plates XII–XIII)

A piece is **alive** if some input needs it (importance above 0.1). From 20, five survive. For each feature, find the
piece that is most important when the feature is on alone, and compare its column with the feature's arrow:
**MMCS** (mean cosine, 1 = same direction) and **ML2R** (mean length ratio, 1 = same size).

In [ ]:
with torch.no_grad():
    xe = batch(100000, device=GPU)
    tgt, ins = run(xe, on=())
    lo1, _ = importance("layer 1", ins["layer 1"] @ A)
    lo2, _ = importance("layer 2", ins["layer 2"] @ B.T)
    alive = [int((lo > 0.1).any(0).sum()) for lo in (lo1, lo2)]
    masked = ((run(xe, {"layer 1": lo1, "layer 2": lo2})[0] - tgt) ** 2).mean().item()
    cols = torch.einsum("jc,ch->cjh", A, B)                         # piece c's column for feature j
    cos = F.cosine_similarity(cols, Wg.T[None], dim=-1)             # 20 x 5
    best = cos.argmax(0)
    mmcs = cos.max(0).values.mean().item()
    ml2r = (cols[best, torch.arange(5)].norm(dim=-1) / Wg.norm(dim=0)).mean().item()
    one = importance("layer 1", torch.eye(5, device=GPU) @ A)[0]     # one feature at a time, value 1
    owners = one.argmax(1).tolist()
    sizes = (A.norm(dim=0) * B.norm(dim=1)).cpu()
print(f"alive: {alive[0]} pieces in layer 1, {alive[1]} in layer 2; each feature's own piece: {owners}")
print(f"MMCS {mmcs:.4f}, ML2R {ml2r:.4f}; piece sizes {sorted([round(v, 3) for v in sizes.tolist()], reverse=True)[:7]} ...")
print(f"output error with every piece scaled by its importance: {masked:.2e} (predicting zero: {(tgt ** 2).mean():.2e})")
check("live pieces, layer 1", alive[0], 5)
check("live pieces, layer 2", alive[1], 5)
check("features with their own piece", len(set(owners)), 5)
check("MMCS", mmcs, 1.0, tol=1e-3)
check("ML2R (the lecture's five seeds: 0.990-0.996)", ml2r, sum([0.9898350834846497, 0.9962183237075806]) / 2, tol=0.006)
check("masked-output error (the lecture's five seeds: 2.5-2.7e-4)", masked, sum([0.0002514039515517652, 0.0002731971035245806]) / 2, tol=6e-5)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
Wc = Wg.cpu()
for j in range(5):
    a1.annotate("", Wc[:, j].tolist(), (0, 0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=4, alpha=0.45))
    a1.annotate("", cols[owners[j], j].cpu().tolist(), (0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1))
a1.set_xlim(-1.4, 1.4); a1.set_ylim(-1.4, 1.4); a1.set_aspect("equal")
a1.set_title("W's columns (wide, grey); each feature's piece (black)", fontsize=9, loc="left")
by_size = sizes.argsort(descending=True)
a2.bar(range(20), sizes[by_size], color=INK)
a2.set_xticks(range(20), [str(int(c)) for c in by_size], fontsize=7)
a2.set_xlabel("piece"); a2.set_ylabel("size (‖A‖ × ‖B‖)"); a2.set_title("the 20 pieces after training", fontsize=9, loc="left")
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot([h_[0] for h_ in history], [h_[1] for h_ in history], color=INK, marker="o", ms=3, label="ours")
ax.plot([s for s, _ in [[0, 13], [2000, 12], [4000, 14], [6000, 13], [8000, 10], [10000, 7], [12000, 5], [14000, 5], [16000, 5], [18000, 5], [20000, 5], [22000, 5], [24000, 5], [26000, 5], [28000, 5], [30000, 5], [32000, 5], [34000, 5], [36000, 5], [38000, 5], [40000, 5]]], [a for _, a in [[0, 13], [2000, 12], [4000, 14], [6000, 13], [8000, 10], [10000, 7], [12000, 5], [14000, 5], [16000, 5], [18000, 5], [20000, 5], [22000, 5], [24000, 5], [26000, 5], [28000, 5], [30000, 5], [32000, 5], [34000, 5], [36000, 5], [38000, 5], [40000, 5]]], color=INK2, ls="--", label="the lecture's (seed 0, CPU)")
ax.set_xlabel("step"); ax.set_ylabel("live pieces"); ax.legend(fontsize=9); plt.show()

Twenty random pieces, five survivors, each one feature's arrow. The other fifteen shrink to almost nothing, yet all twenty
still add up to $W$. Our run is on a GPU, so its random numbers differ from the lecture's CPU run; the outcome is the
same. (On this toy, the lecture's five seeds all came out clean.)

### The importance matrix, live, and surgery on one piece (Plates XIV–XV)

From here we use the lecture's own trained pieces (seed 0), downloaded, so every number can be compared exactly. In layer
1 each feature needs exactly its own piece; in layer 2 a feature at full strength also needs its two neighbours' pieces,
because at value 1 the model leaks into its neighbours: importance reports what the network *does*, interference included.

In [ ]:
P = DATA["tms52_pieces"]
LA, LB = torch.tensor(P["A"]), torch.tensor(P["B"])
LG = {k: [torch.tensor(P["gates"][s][n]) for n in ("w_in", "b_in", "w_out", "b_out")]
      for k, s in (("layer 1", "linear1"), ("layer 2", "linear2"))}

def L_importance(layer, reads):
    w_in, b_in, w_out, b_out = LG[layer]
    z = (F.gelu(reads[..., None] * w_in + b_in) * w_out).sum(-1) + b_out
    return torch.where(z > 0, z.clamp(max=1), 0.01 * z)

def L_run(x, m1=None, m2=None):
    """The lecture's pieces as a model; m1, m2 scale each piece in layers 1 and 2 (None: all fully on)."""
    c = x @ LA
    h = (c * m1 if m1 is not None else c) @ LB
    c2 = h @ LB.T
    return F.relu((c2 * m2 if m2 is not None else c2) @ LA.T + b)

def L_ci(x):
    return L_importance("layer 1", x @ LA), L_importance("layer 2", (x @ LA @ LB) @ LB.T)

for v in ("1.0", "0.25"):
    ci1, ci2 = L_ci(torch.eye(5) * float(v))
    lec = P["ci_onefeature"][v]
    print(f"one feature at {v}: its own layer-1 piece scores", [round(ci1[j, P["owner"][j]].item(), 2) for j in range(5)])
    check(f"importance matrices at value {v}, against the lecture's", max((ci1 - torch.tensor(lec["linear1"])).abs().max().item(),
          (ci2 - torch.tensor(lec["linear2"])).abs().max().item()), 0, tol=2e-3)

from matplotlib.colors import LinearSegmentedColormap
ci1, ci2 = L_ci(torch.eye(5))
fig, axes = plt.subplots(2, 1, figsize=(8, 3.4))
for ax, m, lab in zip(axes, (ci1, ci2), ("layer 1", "layer 2")):
    ax.imshow(m[:, P["order"]].clamp(0, 1), cmap=LinearSegmentedColormap.from_list("ink", [PAPER, INK]), vmin=0, vmax=1,
              aspect="auto")
    ax.set_yticks(range(5), [f"feature {j}" for j in range(5)], fontsize=8)
    ax.set_xticks(range(20), [str(c) for c in P["order"]], fontsize=7)
    ax.set_title(f"{lab}: how much each piece is needed, each feature alone at value 1", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

x = torch.tensor([0.8, 0, 0.6, 0, 0])
ci1, ci2 = L_ci(x)
k = P["owner"][2]                                              # feature 2's piece (the same piece in both layers)
off1, off2 = ci1.clone(), ci2.clone()
off1[k] = off2[k] = 0
print("\nthe model:                      ", [round(v, 4) for v in tms(x, W, b).tolist()])
print("pieces at their importance:     ", [round(v, 4) for v in L_run(x, ci1, ci2).tolist()])
print(f"piece {k} (feature 2's) switched off:", [round(v, 4) for v in L_run(x, off1, off2).tolist()])
gap = lambda a, b_: max(abs(u - v) for u, v in zip(a.tolist(), b_))
check("feature 2's piece switched off: largest gap from the lecture's output", gap(L_run(x, off1, off2), P["ablation"]["ablated_out"]), 0, tol=2e-4)
check("every piece at its importance: largest gap", gap(L_run(x, ci1, ci2), P["ablation"]["ci_masked_out"]), 0, tol=2e-4)

Switch off feature 2's piece and output 2 vanishes, and output 0 drops by 0.243: exactly feature 2's leak, 0.6 × 0.404.
The piece carries the feature's own output *and* its interference. One honest caveat (Plate XV): with every piece scaled
to its importance, the outputs are 1.075 and 0.940, not the model's 1.037 and 0.846; that is SPD's reconstruction error
on this input (it averages 2.7 × 10⁻⁴ over inputs).

---
## Part the Third: beyond the pentagon

### Bigger, and a mechanism that is not a feature (Plates XVII–XVIII)

The lecture also ran 40 features in 10 neurons (200 pieces) and the pentagon with an extra **identity** layer in the
middle, $\text{out} = \mathrm{ReLU}(W^\top I W x + b)$. Those runs are larger (a T4 for 10 minutes each); here are their
saved results, read and recomputed.

In [ ]:
T40 = DATA["tms4010"]
print(f"40 features in 10 neurons (seed 0): MMCS {T40['mmcs']:.4f}, ML2R {T40['ml2r']:.4f}, "
      f"{len(set(T40['owner']))} of 40 features with their own piece, {T40['alive']['linear1']} live pieces of 200")
check("40-feature run: features with their own piece", len(set(T40["owner"])), 39)

ID = DATA["tms52id"]
AH, BH = torch.tensor(ID["AH"]), torch.tensor(ID["BH"])
size = AH.norm(dim=0) * BH.norm(dim=1)
big = size > 0.1 * size.max()
I_pieces = (AH[:, big] @ BH[big]).T
print(f"the identity layer splits into {int(big.sum())} large pieces (sizes {[round(v, 3) for v in size[big].tolist()]}); "
      f"the next largest is {size[~big].max():.3f}. Their sum:\n{I_pieces.numpy().round(3)}")
check("large pieces of the identity", int(big.sum()), 2)
check("their sum: largest gap from the lecture's", (I_pieces - torch.tensor(ID["identity"]["sum_big"])).abs().max().item(), 0, tol=1e-3)

The 2 × 2 identity becomes two pieces, and nearly every input needs them: a mechanism every input uses, which is not a
feature, and which no prism on a single activation could express.

### The prism's view (Plates XIX–XX)

Now put a **prism** (a sparse autoencoder, Lecture 17) on the pentagon's two neurons, trained as the lecture trained it:
a ReLU encoder, unit-length decoder rows, an L1 penalty λ on the activations. First weak sparsity, 5 latents, λ = 0.003,
on the CPU with the lecture's seed.

In [ ]:
def train_prism(m, lam, seed=0, steps=20000):
    torch.manual_seed(seed)
    We = (torch.randn(m, 2) / math.sqrt(2)).requires_grad_(True)
    be = torch.zeros(m, requires_grad=True)
    Wd = torch.randn(2, m); Wd = (Wd / Wd.norm(dim=0, keepdim=True)).requires_grad_(True)
    bd = torch.zeros(2, requires_grad=True)
    opt = torch.optim.Adam([We, be, Wd, bd], lr=1e-3)
    for _ in range(steps):
        hh = batch(4096) @ W.T
        f = F.relu((hh - bd) @ We.T + be)
        loss = ((f @ Wd.T + bd - hh) ** 2).sum(-1).mean() + lam * f.sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            Wd /= Wd.norm(dim=0, keepdim=True)
    with torch.no_grad():
        x = batch(200000)
        x = x[x.abs().sum(-1) > 0]
        hh = x @ W.T
        f = F.relu((hh - bd) @ We.T + be)
        rec = f @ Wd.T + bd
        live = (f > 1e-6).float().mean(0) > 1e-3
        cosm = F.cosine_similarity(Wd.T[:, None, :], W.T[None], dim=-1)
        return {"fvu": (((rec - hh) ** 2).sum() / ((hh - hh.mean(0)) ** 2).sum()).item(), "alive": int(live.sum()),
                "angles": [round(math.degrees(math.atan2(Wd[1, i].item(), Wd[0, i].item())), 1) for i in range(m) if live[i]],
                "per_feature": [int(((cosm.argmax(1) == j) & live & (cosm.max(1).values > 0.9)).sum()) for j in range(5)],
                "out_mse": ((tms(x, W, b) - F.relu(rec @ W + b)) ** 2).mean().item(), "Wd": Wd}
t = time.time()
weak = train_prism(5, 0.003)
print(f"({time.time() - t:.0f} s) 5 latents, λ 0.003: {weak['alive']} live, angles {weak['angles']}, "
      f"unexplained {weak['fvu']:.1e}")
check("live latents", weak["alive"], 4)
check("largest angle difference from the lecture's", max(abs(a - b_) for a, b_ in zip(sorted(weak["angles"]), sorted([44.7, 137.2, -42.6, -135.2]))), 0, tol=1.0)
fig, ax = plt.subplots(figsize=(4, 4))
for j in range(5):
    ax.annotate("", W[:, j].tolist(), (0, 0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1))
for a in weak["angles"]:
    r_ = math.radians(a)
    ax.annotate("", (1.25 * math.cos(r_), 1.25 * math.sin(r_)), (0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.8))
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal")
ax.set_title("grey: the features; black: the prism's live latents", fontsize=9, loc="left"); plt.show()

A perfect reconstruction, from the four **diagonals**, not the pentagon: two numbers can always be rebuilt from four
diagonal arrows. Only feature 0, at 44.7°, happens to sit on one. Make the prism sparser (10 latents, λ = 0.3) and it
finds the features, at a price:

In [ ]:
t = time.time()
strong = train_prism(10, 0.3)
print(f"({time.time() - t:.0f} s) 10 latents, λ 0.3: {strong['alive']} live, latents per feature {strong['per_feature']}, "
      f"unexplained {strong['fvu']:.1%}")
print(f"the model's output with the prism's reconstruction in the neurons: error {strong['out_mse']:.1e}, "
      f"{strong['out_mse'] / 0.0002731971035245806:.0f}x SPD's {0.0002731971035245806:.1e}")
check("every feature found", strong["per_feature"], [1, 1, 1, 1, 1])
check("unexplained share", strong["fvu"], 0.09595249593257904, tol=0.003)
check("output error with the prism", strong["out_mse"], 0.008232252672314644, tol=3e-4)

To read five features out of two numbers, an encoder must accept crosstalk; the model itself leaks above 0.57. The prism
has to *invent* an encoder, and settles for a reconstruction basis or pays in error. SPD never builds an encoder: it asks
which pieces of the model's own weights the output needs.

### Too weak, too strong, too few (Plate XXI)

SPD has its own knobs. The lecture's sweep, on the pentagon:

In [ ]:
print(" β        C    MMCS    ML2R   live pieces")
for r_ in DATA["sweep"]["runs"]:
    print(f"{r_['coeff']:<8g} {r_['C']:3d}   {r_['mmcs']:.3f}   {r_['ml2r']:.3f}   {r_['alive']['linear1']}")

Clean in a window of about 3× around the paper's β. Below it, features split across pieces; above it, they merge, then
everything collapses; and with exactly 5 pieces (C = 5) one feature is lost.

### 100 jobs in 50 neurons (Plate XXII)

The papers' compressed-computation toy: 100 sparse inputs, one MLP of only 50 ReLU neurons, each input's job a little
nonlinear function of itself. SPD splits the first weight matrix, W_in, into 100 pieces, one per input. The lecture's
run, read:

In [ ]:
R = DATA["resid"]
need = torch.tensor(R["export"]["need"]).abs()                 # 100 x 50: how much of each feature W_in sends to each neuron
spread = need > 0.1 * need.max(1, keepdim=True).values
per_feature, per_neuron = spread.sum(1).float(), spread.sum(0).float()
print(f"W_in: {R['export']['distinct']} of 100 features with their own piece; alignment {np.mean(R['owner_cos']):.4f} "
      f"(lowest {min(R['owner_cos']):.4f})")
print(f"each feature's piece spreads over {per_feature.mean():.2f} of the 50 neurons; each neuron serves "
      f"{int(per_neuron.min())}-{int(per_neuron.max())} features (mean {per_neuron.mean():.1f})")
print(f"W_out: {R['alive']['mlp_out']} live pieces; a one-feature input switches on {np.mean(R['pieces_per_feature_out']):.1f} of them")
check("features with their own W_in piece", R["export"]["distinct"], 100)
check("neurons per feature", per_feature.mean().item(), 15.86, tol=0.01)
check("features per neuron, mean", per_neuron.mean().item(), 31.7, tol=0.05)

The neurons are polysemantic (each serves about 30 features); the pieces are not. That is the promise of taking apart the
weights rather than the activations. Where it stands (Plate XXIII): APD (January 2025), SPD (June 2025), and VPD (Goodfire,
May 2026), which reached a 67-million-parameter language model with 38,912 pieces, about 205 active per token (their
numbers).

---
## Your turn

1. **Feature 2 alone.** With the lecture's pieces, put feature 2 alone at 0.6, then switch its piece off. What is left?
   *Hint: `L_ci`, `L_run`, `P["owner"][2]`.*
2. **Graded, not binary.** At value 0.25, how important is each feature's own layer-1 piece?
   *Hint: `L_ci(torch.eye(5) * 0.25)`.*
3. **A bigger prism.** Train a 20-latent prism at λ = 0.003. How many latents live, and do any features get two?
   *Hint: `train_prism(20, 0.003)`.*
4. **Pieces per input.** On random inputs that are not all zero, how many pieces does layer 1 need on average, against the
   number of features that are on? *Hint: `lo1`, `xe` from Part Two.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

---
## Solutions

In [ ]:
# 1. Feature 2 alone
x2 = torch.tensor([0, 0, 0.6, 0, 0])
c1, c2 = L_ci(x2)
k = P["owner"][2]
c1[k] = c2[k] = 0
print("the model:", [round(v, 4) for v in tms(x2, W, b).tolist()], " piece off:", [round(v, 4) for v in L_run(x2, c1, c2).tolist()])
check("feature 2 alone, its piece off: largest gap from the lecture's", gap(L_run(x2, c1, c2), P["ablation_single"]["ablated_out"]), 0, tol=2e-4)

In [ ]:
# 2. Graded, not binary
ci1, _ = L_ci(torch.eye(5) * 0.25)
own = [ci1[j, P["owner"][j]].item() for j in range(5)]
print("own piece at 0.25:", [round(v, 2) for v in own])
check("lowest", min(own), 0.14, tol=0.01)
check("highest", max(own), 0.34, tol=0.01)

In [ ]:
# 3. A bigger prism
big_ = train_prism(20, 0.003)
print(f"20 latents, λ 0.003: {big_['alive']} live; latents per feature {big_['per_feature']}; unexplained {big_['fvu']:.1e}")
check("live latents", big_["alive"], 8, tol=1)
check("latents per feature", big_["per_feature"], [1, 2, 2, 1, 0])

In [ ]:
# 4. Pieces per input
with torch.no_grad():
    nz = xe.abs().sum(-1) > 0
    pieces_needed = (lo1[nz] > 0.1).float().sum(-1).mean().item()
    features_on = (xe[nz] > 0).float().sum(-1).mean().item()
print(f"layer 1 needs {pieces_needed:.2f} pieces per non-zero input; {features_on:.2f} features are on")
check("layer-1 pieces per non-zero input (the lecture: 0.85)", pieces_needed, 0.85, tol=0.1)
print(f"\nthe whole notebook: {(time.time() - T0) / 60:.1f} minutes")